In [2]:
# Install the Telethon library for Telegram API interactions
#!pip install -q telethon
# Telegram imports
from telethon.sync import TelegramClient

# Initial imports
from datetime import datetime, timezone
import time
import json
import os
import errno
import logging
import teleGraph as tg
import pandas as pd

In [3]:
# Directory where you want to store scraped data: 
homedir = 'telegraph_project'
try:
    os.mkdir(homedir)
    os.mkdir(f'{homedir}/scraped_data')
except OSError as e:
    if e.errno != errno.EEXIST:
        raise
utilspath = f'./{homedir}/'
respath = f'./{homedir}/scraped_data/'

In [4]:
#logger config
logger = logging.getLogger(__name__)
logging.basicConfig(filename=utilspath + 'telescrap.log', filemode='w', level=logging.WARNING)

In [5]:
# TELEGRAM CLIENT

#  Your Telegram account username (without '@', e.g., 'johndoe'):
username = 'username' # @param {type:"string"}
# Your Telegram account phone number in string format
phone = '+48111222333' # @param {type:"string"}
#  Your API ID, taken from https://my.telegram.org/apps:
api_id = '11111111' # @param {type:"string"}
# Your API hash, also from https://my.telegram.org/apps:
api_hash = 'apihash' # @param {type:"string"}

In [8]:
# SCRAPING PARAMETERS

# names of the channels or groups that you want to scrape, separated by commas (,).  Write the channel names with '@' or provide a full link, 
# for example: '@volna_lt' or 'https://t.me/volna_lt'. 
channels_list = 'channels_list.data' # @param {type:"string"}
with open(utilspath + f'{channels_list}', "r") as f:
    channels = f.read().splitlines()
#dictionary storing users in a format: username:{'entity_type':entity_type, 'entity_id':entity_id}
#used for caching users to avoid calling Telegram API Request
peers_data_file = 'peers_data.json' # @param {type:"string"}
#check if premade version exists:
try:
    with open(utilspath + f'{peers_data_file}', 'r', encoding='utf-8') as f:
        peers_data = json.loads(f.read())
#if not, create an empty dictionary for this session:
except Exception as e:
    peers_data = dict()
#  Here you can select the time window you would like to extract data from the listed communities:
date_min = '2025-12-15 00:00:00' # @param {type:"date"}
date_max = '2026-04-12 23:59:59' # @param {type:"date"}
date_min = datetime.fromisoformat(date_min).replace(tzinfo=timezone.utc)
date_max = datetime.fromisoformat(date_max).replace(tzinfo=timezone.utc)
#Keyword to search, leave empty if you want to extract all messages from the channel(s):
key_search = '' # @param {type:"string"}
# Maximum number of messages to scrape (only use if you want a specific limit, otherwise leave a high number to scrape everything):
max_t_index = 10**23   # @param {type:"integer"}
# Timeout in seconds, for Google Colab use at most 6 hours, that is 21600 seconds, as Google Colab deactivates itself after that time:
time_limit = 21600*10000 # @param {type:"integer"}
# Choose the format of the final file you want to download: `excel` or `parquet`:
File = 'excel' # @param ["excel", "parquet", "csv"]

In [9]:
# SCRAPING PROCESS

# During this step, Telegram may request a verification code.
# Please monitor your Telegram app and input the required information promptly. Rest assured, all data entered remains secure.
t_index = 0  # Tracker for the number of messages processed
start_time = time.time()  # Record the start time for the scraping session

for channel in channels:
    if t_index >= max_t_index:
        break
    if time.time() - start_time > time_limit:
        break
    loop_start_time = time.time()
    try:
        c_index = 0 # number of scraped messages in channel 
        used_ids = set() # set of unique messages in channel
        data = set() # List to store scraped data
        print(f'\n\n##### {channel} scrapping begin,  {len(used_ids):05} posts already scraped #####\n\n')
        async with TelegramClient(username, api_id, api_hash) as client:
            #iterate over all messages: 
            async for message in client.iter_messages(channel, search=key_search):
                if message.id in used_ids:
                    continue 
                try:
                    if date_min <= message.date <= date_max:
                        #process the message itself:
                        msg = await tg.extract_data_from_message(client, message, channel, None, peers_dict = peers_data)
                        used_ids.add(message.id)
                        data.add(tuple(msg.items()))  
                        #comment/replies processing:
                        if message.replies != None and message.replies.replies > 0:
                            async for comment_message in client.iter_messages(channel, reply_to=message.id):
                                try:
                                    if comment_message.id in used_ids:
                                        continue 
                                    msg = await tg.extract_data_from_message(client, comment_message, channel, message.id, peers_dict = peers_data)
                                    used_ids.add(comment_message.id)
                                    data.add(tuple(msg.items()))
                                except Exception as e:
                                    logger.error('Error processing comment:')
                                    logger.error(e, stack_info=True, exc_info=True) 
                                    print(f'Error processing comment: {e}')
                        c_index += 1
                        t_index += 1
                        # Print progress
                        if t_index % 1000 == 0:
                            print(f'{"-" * 80}')
                            tg.print_progress(t_index, message.id, start_time, max_t_index)
                            current_max_id = min(c_index + message.id, max_t_index)
                            print(f'From {channel}: {c_index:05} contents of {current_max_id:05}')
                            print(f'Id: {message.id:05} / Date: {msg['Date']}')
                            print(f'Total: {t_index:05} contents until now')
                            print(f'{"-" * 80}\n\n')
                        # save partial results
                        if t_index % 10**6 == 0:
                            #save messages:
                            channel_id = peers_data[channel]['entity_id'] if channel in peers_data else ''
                            backup_filename = f'backup_data_msg_until_{t_index:05}_post_ID{message.id:07}_{channel}_{channel_id}'
                            tg.save_data(pd.DataFrame([dict(m) for m in data]), respath + backup_filename, File)  
                        if t_index >= max_t_index:
                            break
                        if time.time() - start_time > time_limit:
                            break
                    elif message.date < date_min:
                        break
                except Exception as e:
                    logger.error('Error processing message:')
                    logger.error(e) 
                    print(f'Error processing message: {e}')
            #save final results:
            print(f'\n\n##### {channel} was ok with {c_index:05} posts #####\n\n')
            dt = [dict(m) for m in data]
            df = pd.DataFrame(dt)
            #save messages
            channel_id = peers_data[channel]['entity_id'] if channel in peers_data else ''
            full_filename = respath + f'complete_{channel}_{channel_id}_msg_data'
            tg.save_data(df, respath + full_filename, File)
    except Exception as e:
        logger.error(f'{channel} error:')
        logger.error(e) 
        print(f'{channel} error: {e}')
    loop_end_time = time.time()
    loop_duration = loop_end_time - loop_start_time
    if loop_duration < 60:
        time.sleep(60 - loop_duration)
#save peers data to json file for further use:
with open(utilspath + f'peers_data.json', 'w', encoding='utf-8') as f:
    json.dump(peers_data, f, ensure_ascii=False, indent=4, default=list)



##### @n_aujenoschat scrapping begin,  00000 posts already scraped #####


--------------------------------------------------------------------------------
Progress: 1.34% | Elapsed Time: 00:00:01:56 | Remaining Time: 00:02:23:17
From @n_aujenoschat: 01000 contents of 74731
Id: 73731 / Date: 2026-03-10 17:18:22
Total: 01000 contents until now
--------------------------------------------------------------------------------


--------------------------------------------------------------------------------
Progress: 2.68% | Elapsed Time: 00:00:02:57 | Remaining Time: 00:01:47:37
From @n_aujenoschat: 02000 contents of 74723
Id: 72723 / Date: 2026-02-25 13:37:08
Total: 02000 contents until now
--------------------------------------------------------------------------------


--------------------------------------------------------------------------------
Progress: 4.02% | Elapsed Time: 00:00:04:17 | Remaining Time: 00:01:42:36
From @n_aujenoschat: 03000 contents of 74709
Id: 71709 / Date:

C:\Users\barto\poliGraph\Lib\site-packages\telethon\utils.py:1566: UserWarning: Using async sessions support is an experimental feature
  warnings.warn('Using async sessions support is an experimental feature')




##### @infalt was ok with 01542 posts #####




##### @n_aujienos scrapping begin,  00000 posts already scraped #####


--------------------------------------------------------------------------------
Progress: 34.27% | Elapsed Time: 00:00:09:42 | Remaining Time: 00:00:18:37
From @n_aujienos: 00077 contents of 15423
Id: 15346 / Date: 2026-03-29 14:17:14
Total: 08000 contents until now
--------------------------------------------------------------------------------




##### @n_aujienos was ok with 00786 posts #####




##### @hmelisozreli scrapping begin,  00000 posts already scraped #####




##### @hmelisozreli was ok with 00185 posts #####




##### @lietuvasu scrapping begin,  00000 posts already scraped #####


@lietuvasu error: No user has "lietuvasu" as username


##### @mkzmedia scrapping begin,  00000 posts already scraped #####


--------------------------------------------------------------------------------
Progress: 28.80% | Elapsed Time: 00:00:23:39 | Remaining Time: 00